# Gold — GA4 date dimension

Python 3.12 notebook (not Spark). Default lakehouse: **GA4Lakehouse** in workspace **GA4-Report**. Attach that lakehouse (Add data items → From OneLake catalog) before running.

Reads `min`/`max` `event_date` from `dbo.bronze_ga4_events` through deltalake (storage bearer token in Fabric; not the raw Files parquet) and DuckDB, then overwrites `dbo.dim_ga4_date` with one row per day in that range. The range is not hardcoded.

`date` is a DATE, the same type as `session_date` / `event_date` in the gold tables (`strptime(CAST(event_date AS VARCHAR), '%Y%m%d')::DATE`). `date_key` is the matching `YYYYMMDD` integer used in bronze `event_date`.

Holiday flags are for this sample's season: Black Friday 2020-11-27, Cyber Monday 2020-11-30, Christmas 2020-12-25, New Year 2021-01-01. `holiday_name` is null on any other day. `is_peak_season` is 27 Nov–31 Dec.

Physical path: `/lakehouse/default/Tables/dbo/dim_ga4_date` (or the equivalent `abfss://` URI).


In [ ]:
# Build dbo.dim_ga4_date from the bronze event_date range (overwrite).
import os
import subprocess
import sys
from pathlib import Path

import pyarrow as pa
from deltalake import DeltaTable, write_deltalake

try:
    import duckdb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb

ROOT = Path(os.environ.get("LAKEHOUSE_DEFAULT", "/lakehouse/default"))
SOURCE = "bronze_ga4_events"
SESSIONS = "gold_ga4_sessions"
TABLE = "dim_ga4_date"


def _storage():
    """abfss + bearer token in Fabric; local filesystem path otherwise."""
    try:
        import notebookutils
        ctx = notebookutils.runtime.context
        ws = ctx["defaultLakehouseWorkspaceId"]
        lh = ctx["defaultLakehouseId"]
        endpoint = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
        base = f"abfss://{ws}@{endpoint}/{lh}/Tables/dbo"
        opts = {
            "bearer_token": notebookutils.credentials.getToken("storage"),
            "use_fabric_endpoint": "true",
        }
        return base, opts
    except Exception:
        dbo = ROOT / "Tables" / "dbo"
        dbo.mkdir(parents=True, exist_ok=True)
        return str(dbo), None


def _cast_null_type(typ: pa.DataType) -> pa.DataType:
    if pa.types.is_null(typ):
        return pa.string()
    if pa.types.is_struct(typ):
        return pa.struct([
            pa.field(f.name, _cast_null_type(f.type), nullable=f.nullable)
            for f in typ
        ])
    if pa.types.is_list(typ) or pa.types.is_large_list(typ) or pa.types.is_fixed_size_list(typ):
        return pa.list_(_cast_null_type(typ.value_type))
    if pa.types.is_map(typ):
        return pa.map_(_cast_null_type(typ.key_type), _cast_null_type(typ.item_type))
    return typ


def _cast_nulls(table: pa.Table) -> pa.Table:
    new_fields, new_cols = [], []
    for i, field in enumerate(table.schema):
        new_type = _cast_null_type(field.type)
        col = table.column(i)
        if new_type != field.type:
            col = col.cast(new_type)
        new_fields.append(pa.field(field.name, new_type, nullable=field.nullable))
        new_cols.append(col)
    return pa.Table.from_arrays(new_cols, schema=pa.schema(new_fields))


def _arrow(con, sql):
    rel = con.sql(sql)
    if hasattr(rel, "to_arrow_table"):
        return rel.to_arrow_table()
    if hasattr(rel, "fetch_arrow_table"):
        return rel.fetch_arrow_table()
    return rel.arrow()


def _one(con, sql):
    rows = con.sql(sql).fetchall()
    return rows[0] if rows else None


def _write(name, table, base, opts):
    uri = f"{base}/{name}"
    write_deltalake(
        uri,
        _cast_nulls(table),
        mode="overwrite",
        schema_mode="overwrite",
        storage_options=opts,
    )
    return uri


base, opts = _storage()
src_uri = f"{base}/{SOURCE}"
print(f"source: {src_uri}")

con = duckdb.connect()
con.register(SOURCE, DeltaTable(src_uri, storage_options=opts).to_pyarrow_dataset())

min_d, max_d = _one(con, f"""
SELECT
    min(strptime(CAST(event_date AS VARCHAR), '%Y%m%d')::DATE),
    max(strptime(CAST(event_date AS VARCHAR), '%Y%m%d')::DATE)
FROM {SOURCE}
""")
if min_d is None or max_d is None:
    raise SystemExit(f"FAIL {TABLE}: no event_date in {SOURCE}")

expected = (max_d - min_d).days + 1
print(f"bronze event_date range: {min_d} to {max_d} ({expected} days)")

dim = _arrow(con, f"""
WITH bounds AS (
    SELECT
        min(strptime(CAST(event_date AS VARCHAR), '%Y%m%d')::DATE) AS min_d,
        max(strptime(CAST(event_date AS VARCHAR), '%Y%m%d')::DATE) AS max_d
    FROM {SOURCE}
),
spine AS (
    SELECT CAST(generate_series AS DATE) AS d, min_d
    FROM bounds, generate_series(min_d, max_d, INTERVAL 1 DAY)
)
SELECT
    d AS date,
    CAST(year(d) * 10000 + month(d) * 100 + day(d) AS INTEGER) AS date_key,
    CAST(year(d) AS INTEGER) AS year,
    CAST(quarter(d) AS INTEGER) AS quarter,
    CAST(month(d) AS INTEGER) AS month_num,
    monthname(d) AS month_name,
    strftime(d, '%b') AS month_short,
    strftime(d, '%Y-%m') AS year_month,
    date_trunc('month', d)::DATE AS month_start,
    CAST(week(d) AS INTEGER) AS iso_week,
    date_trunc('week', d)::DATE AS week_start,
    CAST(day(d) AS INTEGER) AS day_of_month,
    CAST(isodow(d) AS INTEGER) AS day_of_week_num,
    dayname(d) AS day_name,
    strftime(d, '%a') AS day_short,
    isodow(d) >= 6 AS is_weekend,
    d = DATE '2020-11-27' AS is_black_friday,
    d = DATE '2020-11-30' AS is_cyber_monday,
    d = DATE '2020-12-25' AS is_christmas,
    d = DATE '2021-01-01' AS is_new_year,
    CASE
        WHEN d = DATE '2020-11-27' THEN 'Black Friday'
        WHEN d = DATE '2020-11-30' THEN 'Cyber Monday'
        WHEN d = DATE '2020-12-25' THEN 'Christmas'
        WHEN d = DATE '2021-01-01' THEN 'New Year'
    END AS holiday_name,
    (month(d) = 11 AND day(d) >= 27) OR (month(d) = 12) AS is_peak_season,
    CAST(datediff('day', min_d, d) AS INTEGER) AS days_from_start
FROM spine
ORDER BY d
""")

uri = _write(TABLE, dim, base, opts)
print(f"wrote {TABLE}: {uri} rows={len(dim)}")
print(f"dim.date type: {dim.schema.field('date').type}")
con.register(TABLE, DeltaTable(uri, storage_options=opts).to_pyarrow_dataset())

sess_uri = f"{base}/{SESSIONS}"
sess_ds = DeltaTable(sess_uri, storage_options=opts).to_pyarrow_dataset()
con.register(SESSIONS, sess_ds)
print(f"gold_ga4_sessions.session_date type: {sess_ds.schema.field('session_date').type}")

rows, distinct_dates, dupes = _one(con, f"""
SELECT
    COUNT(*)::BIGINT,
    COUNT(DISTINCT date)::BIGINT,
    COUNT(*)::BIGINT - COUNT(DISTINCT date)::BIGINT
FROM {TABLE}
""")
missing_sessions = _one(con, f"""
SELECT COUNT(*)::BIGINT
FROM (SELECT DISTINCT session_date FROM {SESSIONS}) s
ANTI JOIN {TABLE} d ON d.date = s.session_date
""")[0]
span_days, first_d, last_d = _one(con, f"""
SELECT
    datediff('day', min(date), max(date)) + 1,
    min(date),
    max(date)
FROM {TABLE}
""")

print(f"rows={rows:,} distinct_dates={distinct_dates:,} dupes={dupes:,}")
print(f"span={first_d} to {last_d} ({span_days} days)")
print(f"session_dates missing from dim: {missing_sessions:,}")

errors = []
if rows != expected:
    errors.append(f"rows {rows} != (max-min)+1 {expected}")
if rows != span_days:
    errors.append(f"rows {rows} != min-to-max span {span_days}")
if expected != 92:
    errors.append(f"(max-min)+1 is {expected}, expected 92 days in this sample")
if rows != 92:
    errors.append(f"rows {rows} != 92")
if dupes:
    errors.append(f"{dupes} duplicate dates")
if missing_sessions:
    errors.append(f"{missing_sessions} gold_ga4_sessions.session_date values missing from dim")
if str(dim.schema.field("date").type) != str(sess_ds.schema.field("session_date").type):
    errors.append(
        f"date type {dim.schema.field('date').type} != "
        f"session_date type {sess_ds.schema.field('session_date').type}"
    )

if errors:
    msg = "; ".join(errors)
    print(f"FAIL {TABLE}: {msg}")
    raise SystemExit(f"FAIL {TABLE}: {msg}")
print(f"PASS {TABLE}: rows={rows:,} {first_d} to {last_d}")
